# 5.6 阶段四：preload 流水

乒乓解决了「搬运 vs 计算」的等待，但**计算内部**仍是串行的：C1 完才 V1，V1 完才 C2……本节引入 preload 流水，让 Cube 与 Vector 交叉掩盖、减少流水断流。这是 FA 性能最核心的一步。

![三种编排对比](images/preload_pipeline.svg)

## 1. 四拍的依赖关系

先记清楚符号（后文全部使用）：

| 拍 | 内容 |
|--|--|
| **C1** | 计算 $QK^\top$（Cube） |
| **V1** | 计算 scale、mask、FlashSoftmax，得到 $P$（Vector） |
| **C2** | 计算 $PV$（Cube） |
| **V2** | 用当前基本块的 exp_max 更新之前的 flashupdate 结果；将更新后的结果与当前基本块 C2 结果相加；若是 s2 方向最后一个基本块则除以行和，否则写回 UB 等待新一轮更新（Vector） |

依赖链：**V1 依赖 C1 的结果；C2 依赖 V1 的结果；V2 依赖 C2 的结果**。同块内四拍只能串行——但**不同块之间**有调度自由度。

## 2. 时间画像与瓶颈定位

FA 各阶段计算时间大约为：

```text
V1 > C1 ≈ C2 >> V2
```

即 Vector 的 softmax 拍最长，场景整体为 **Cube Bound**（Cube 是瓶颈资源）——而**较长的 V1 阶段容易引起 Cube 流水断流**：C2 必须等 V1 完成，Cube 只能干等。

> 适用性提醒：preload 流水针对 FA 算子中 **Cube Bound 或 Vector Bound 的计算 Bound 场景**；如果是 Memory Bound（搬运受限）场景，则应该用 N Buffer 流水模板。

## 3. 三种编排对比

### ① 串行（阶段三之前）

Cube 和 Vector 完全没有并行，总时长 = 所有拍之和。

### ② N Buffer 流水（N=2）

两组任务交替：第 0 组的 V1 执行时，第 1 组的 C1 已经在 Cube 上跑——有一定重叠，但 **Cube 仍有断流**：较长的 V1 拖住 C2 的发射。

### ③ Preload 流水

> **preload 流水把当前轮 V1 对应的 C2 放到下一轮发射，互换 C1、C2 的位置，降低了 V1→C2 数据依赖的影响。**

效果：**一轮使用两个 Cube 阶段（C1、C2）掩盖一个较长的 Vector 阶段（V1）**，Cube 无断流。伪代码：

```python
# NBuffer 流水
task = 0
for BN2S1S2:
    for i in range(task, task + N):    # 先连发 N 组的 C1/V1
        C1(i); V1(i)
    for i in range(task, task + N):    # 再连发 N 组的 C2/V2
        C2(i); V2(i)
    task += N

# Preload 流水（PRELOAD_NUM 轮）
task = 0
for BN2S1S2:
    C1(task)
    V1(task)
    if task > PRELOAD_NUM - 1:          # 滞后 PRELOAD_NUM 轮再发射 C2/V2
        C2(task - PRELOAD_NUM)
        V2(task - PRELOAD_NUM)
    task += 1
for PRELOAD_NUM:                        # 收尾：清空滞后的尾巴
    C2(task - PRELOAD_NUM)
    V2(task - PRELOAD_NUM)
    task += 1
```

实际工程中，Preload 通过 `taskId` 与 `PRELOAD_N` 参数控制流水节奏（第 3 章 3.7 节的 runInfo 机制）。

## 4. Buffer 数量是不是越多越好？

一个自然的想法：N Buffer(N=3) 的 Cube 流水似乎也没有断流，何不加大 N？

**不行**，因为——

1. **Buffer 的数量直接影响基本块的大小**：片上容量固定，buffer 份数越多，每份越小；
2. **基本块越小，数据块的重复搬运越多**（Q、V 等共享矩阵被更多次重新搬入）；
3. **过小的基本块可能导致搬运带宽打不满**，场景反而变成搬运 Bound。

> 这就是第 3 章 3.4 节「基本块要在交互与效率之间找平衡、需要不停测试找最优解」的具体体现。

## 5. 用 Python 模拟 CV 流水

调流水编排不必每次上真机：可以用 `threading` 模拟 AIC 和 AIV、用 `Event` 做线程间同步，一键生成 N Buffer 与 Preload 的流水图。

```python
import threading, time

# 每个依赖边一个事件：C1→V1、V1→C2、C2→V2
event_C1V1 = {i: threading.Event() for i in range(4)}
event_V1C2 = {i: threading.Event() for i in range(4)}
event_C2V2 = {i: threading.Event() for i in range(4)}

C1_TIME, V1_TIME, C2_TIME, V2_TIME = 3, 8, 3, 1   # V1 > C1≈C2 >> V2

def cube_core(tasks, preload):
    # 按编排规则发射 C1 / C2（串行模拟单 Cube 单元）
    ...

def vector_core(tasks, preload):
    # 按编排规则发射 V1 / V2，wait/set 对应事件
    ...
```

核心技巧：用 `set` / `wait` 模拟同步，**与实际业务代码里的信号量语义一一对应**。

两个思考题（也是工程选型题）：

1. 为什么用多线程而不是多进程？——模拟程序没有实际计算任务，本质不是 CPU bound；Thread 比 Process 更轻量、占用资源少、可启动数目更多，便于扩展。
2. 模拟器的不足？——目前没有模拟搬运单元，运行时间不够真实；改进方向是增加搬运线程，并把搬运/计算速率从真实芯片数据标定。

## 小测验

1. 写出四拍的依赖链。哪种编排能实现「Cube 无断流」？原理是什么？
2. FA 的时间画像 V1 > C1 ≈ C2 >> V2 意味着什么 Bound？V1 长会带来什么问题？
3. Buffer 数量为什么不是越多越好？说出至少两条原因。
4. Preload 伪代码里 `C2(task - PRELOAD_NUM)` 滞后发射的意义是什么？收尾循环为什么必须有？